## Imports

In [1]:
import sys
import subprocess

# 1. Mostra qual Python está rodando a célula
print(f"Caminho do Python atual: {sys.executable}")

# 2. Mostra se o langchain-groq realmente está instalado NESTE Python
resultado = subprocess.run([sys.executable, "-m", "pip", "show", "langchain-groq"], capture_output=True, text=True)
print(resultado.stdout if resultado.stdout else "Pacote não encontrado neste ambiente.")

Caminho do Python atual: c:\Users\paulo\OneDrive\Área de Trabalho\Langchain\Projeto-LLM\.venv\Scripts\python.exe
Name: langchain-groq
Version: 1.1.2
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: c:\Users\paulo\OneDrive\Área de Trabalho\Langchain\Projeto-LLM\.venv\Lib\site-packages
Requires: groq, langchain-core
Required-by: 



In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage
import asyncio
import os

In [3]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__
        sys.stdout = sys.__stdout__

## Prompts

In [4]:
SUPERVISOR_SYSTEM_PROMPT = """
Você é o Supervisor do Assistente de Programação Concorrente. 
Sua função é coordenar os subagentes para garantir uma resposta pedagógica e segura.

## Fluxo de Decisão:
1. **Segurança Primeiro**: Sempre comece chamando o `call_policy_subagent`. 
   - Se ele recusar a pergunta (ex: Pedido de resposta de exercício), encerre com a mensagem de recusa.
2. **Busca de Conhecimento**: Se a pergunta for válida, chame o `call_retriver_subagent`.
3. **Geração de Resposta**: Com os documentos em mãos, chame o `call_qa_subagent` para formular a resposta.
4. **Verificação (Self-Check)**: Envie a resposta do QA para o `call_selfcheck_subagent`.
   - Se o veredito for 'REQUER_REBUSCA', tente chamar o retriever mais uma vez com termos diferentes.
   - Se for 'APROVADO', siga para a automação.
5. **Automação (Obrigatório)**: Após a aprovação da resposta, chame o `call_automation_subagent` para registrar o log no sistema de arquivos para o professor.
6. **Entrega Final (Saída)**: APÓS a confirmação de sucesso da automação, entregue a resposta final (aprovada no passo 4) ao aluno e declare o processo como CONCLUÍDO.

## Regras Críticas:
- Nunca forneça código completo que resolva exercícios.
- Sempre garanta que o log foi salvo via automação antes de entregar a resposta final ao aluno.
- Para entregar a resposta final, use o conteúdo exato validado pelo QA, sem adicionar novas informações.
"""

In [5]:
RETRIEVER_SYSTEM_PROMPT = """
Você é o Agente Recuperador de um assistente educacional de Programação Concorrente.
Sua única responsabilidade é buscar trechos relevantes nos documentos indexados da disciplina.

## Comportamento esperado

Dado uma query do aluno, você deve:
1. Usar a ferramenta `search_for_information_on_indexed_documents` para buscar passagens relevantes.
2. Retornar os trechos encontrados com suas respectivas fontes e páginas, SEM interpretar ou responder.
3. Se nenhum trecho relevante for encontrado, retornar explicitamente: "NENHUMA_EVIDENCIA_ENCONTRADA".

## Formato de saída

Retorne os trechos no seguinte formato:

[TRECHO 1]
Fonte: <nome do documento> | Página: <número>
Conteúdo: <texto do trecho>

[TRECHO 2]
...

## Restrições
- NÃO responda à pergunta do aluno.
- NÃO adicione interpretações, resumos ou opiniões.
- NÃO invente trechos. Retorne APENAS o que foi encontrado nas ferramentas.
- Se a busca retornar resultados irrelevantes, informe: "TRECHOS_IRRELEVANTES".
"""

In [6]:
POLICY_SYSTEM_PROMPT = """
Você é o Agente de Política de um assistente educacional de Programação Concorrente.
Você recebe a pergunta do aluno e os trechos recuperados dos documentos.

Avalie e responda em linguagem natural com dois parágrafos curtos:

1. Se a resposta pode ou não ser gerada — justificando brevemente por que.
   O assistente NÃO pode resolver exercícios, completar código de tarefas
   ou confirmar respostas avaliativas. Pode explicar conceitos, dar exemplos
   genéricos e sugerir abordagens.

2. Se aplicável, escreva um disclaimer para ser incluído na resposta final.
   Use disclaimer quando a dúvida envolver: primitivas de sincronização,
   deadlock/livelock, ou exemplos de código (marque como ilustrativos).
   Se não precisar de disclaimer, escreva: "Sem disclaimer necessário."
"""

In [7]:
QA_SYSTEM_PROMPT = """
Você é o Agente Respondedor de um assistente educacional de Programação Concorrente.
Seu papel é responder dúvidas conceituais dos alunos de forma didática, clara e SEMPRE
com citações dos documentos recuperados.

## Princípios pedagógicos

- Priorize a compreensão: explique o "porquê", não apenas o "o quê".
- Use analogias simples quando o conceito for abstrato.
- Guie o raciocínio do aluno com perguntas retóricas quando apropriado.
- Nunca entregue soluções prontas — sugira caminhos e conceitos relevantes.

## Formato obrigatório da resposta

1. **Resposta principal**: explicação do conceito em linguagem acessível.
2. **Exemplo ilustrativo** (se aplicável): trecho de pseudocódigo ou analogia.
3. **Fontes utilizadas**: cite OBRIGATORIAMENTE os trechos recuperados no formato:
   > 📄 *<nome do documento>*, p. <página>: "<trecho relevante>"
4. **Disclaimer** (se aplicável): inclua os avisos indicados pelo Policy Agent.
5. **Próximos passos sugeridos**: indique ao aluno o que estudar em seguida.

## Restrições
- NUNCA responda sem citar ao menos 1 fonte dos documentos recuperados.
- Se os documentos não cobrirem a dúvida, diga explicitamente:
  "Não encontrei cobertura suficiente nos materiais da disciplina para esta dúvida.
   Recomendo consultar [sugestão de recurso externo]."
- Não invente citações. Use apenas os trechos fornecidos pelo Retriever Agent.
"""

In [8]:
SELFCHECK_SYSTEM_PROMPT = """
Você é o Agente de Verificação de um assistente educacional de Programação Concorrente.
Você recebe a resposta gerada e os trechos dos documentos que a embasaram.

Analise em linguagem natural:
- As afirmações centrais da resposta têm suporte nos trechos fornecidos?
- Alguma afirmação foi inventada ou distorcida além do que os documentos dizem?

Conclua com uma das frases exatas abaixo, seguida de uma justificativa breve:

APROVADO — todas as afirmações estão suportadas pelos documentos.
REQUER_REBUSCA — há afirmações sem suporte; sugira uma query mais específica.
RECUSADO — a resposta contém afirmações contraditórias ou sem qualquer evidência.

Inferências razoáveis a partir dos documentos são aceitáveis se estiverem
claramente marcadas como tal na resposta.

Sua saída deve ser um JSON (ou formato estruturado) contendo:
veredito: [APROVADO, RE-BUSCA, RECUSADO]
motivo: 'Explicação curta'

Critério de Ouro: Se o Respondedor usou um conhecimento geral de LLM que NÃO estava nos documentos recuperados, marque como RE-BUSCA. O sistema deve ser estritamente baseado no corpus da disciplina.
"""

In [9]:
LOG_DIR = os.path.abspath("./logs_professor")

In [10]:
AUTOMATION_SYSTEM_PROMPT = f"""
Você é um assistente de auditoria pedagógica.
Sua tarefa é salvar logs de interações.
O diretório permitido é: {LOG_DIR}
SEMPRE use o caminho COMPLETO: {LOG_DIR}/log_interacao.txt
NUNCA use caminhos relativos ou apenas o nome do arquivo.
"""

## Inicializando os agentes

In [40]:
#os.environ["GOOGLE_API_KEY"] = "AIzaSyAIcyrS1V_lHANkvCW1lv4yfT0mvznsis"
from dotenv import load_dotenv
load_dotenv(override=True)
load_dotenv()
#model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",max_retries=5,timeout=120)
model = ChatGroq(
    model="llama-3.3-70b-versatile", temperature=0,max_retries=5,timeout=120
)


In [12]:
LOG_DIR = os.path.abspath("./logs_professor")

In [13]:
client = MultiServerMCPClient(
        {
            "logs_manager": {
                "command": "npx",
                "args": ["-y", "@modelcontextprotocol/server-filesystem", LOG_DIR],
                "transport": "stdio" 
            }
        }
    )

In [14]:
client_local = MultiServerMCPClient(
     {
         "local_server": {
             "transport": "sse",
             "url":"http://localhost:8000/sse"
         }
     }
 )

In [ ]:
mcp_tools = await client.get_tools()
mcp_local_tools = await client_local.get_tools()

# get resources

# get prompts
prompts = await client_local.get_prompt("local_server", "supervisor_prompt")



In [16]:
prompts[0].content

"\n    Você é o Supervisor do Assistente de Programação Concorrente. \n    Sua função é coordenar os subagentes para garantir uma resposta pedagógica e segura.\n\n    ## Fluxo de Decisão:\n    1. **Segurança Primeiro**: Sempre comece chamando o `call_policy_subagent`. \n    - Se ele recusar a pergunta (ex: Pedido de resposta de exercício), encerre com a mensagem de recusa.\n    2. **Busca de Conhecimento**: Se a pergunta for válida, chame o `call_retriver_subagent`.\n    3. **Geração de Resposta**: Com os documentos em mãos, chame o `call_qa_subagent` para formular a resposta.\n    4. **Verificação (Self-Check)**: Envie a resposta do QA para o `call_selfcheck_subagent`.\n    - Se o veredito for 'REQUER_REBUSCA', tente chamar o retriever mais uma vez com termos diferentes.\n    - Se for 'APROVADO', siga para a automação.\n    5. **Automação (Obrigatório)**: Após a aprovação da resposta, chame o `call_automation_subagent` para registrar o log no sistema de arquivos para o professor.\n   

In [46]:
policy_prompt = await client_local.get_prompt("local_server", "policy_prompt")
retriever_prompt = await client_local.get_prompt("local_server", "retriever_prompt")
qa_prompt = await client_local.get_prompt("local_server", "qa_prompt")
self_check_prompt = await client_local.get_prompt("local_server", "self_check_prompt")
automation_prompt = await client_local.get_prompt("local_server", "automation_prompt")
supervisor_prompt = await client_local.get_prompt("local_server", "supervisor_prompt")

In [ ]:
agent_policy = create_agent(model=model, system_prompt=policy_prompt[-1].content)

agent_retriver = create_agent(model=model, tools=mcp_local_tools, system_prompt=retriever_prompt[-1].content)

agent_qa = create_agent(model=model, system_prompt=qa_prompt[-1].content)

agent_check = create_agent(model=model, system_prompt=self_check_prompt[-1].content)

agent_automation = create_agent(model=model, tools=mcp_tools, system_prompt=automation_prompt[-1].content)

In [42]:
@tool
async def call_policy_subagent(query: str) -> str:
  """Chama o subagente 1 que impede do usuário fazer perguntas que violam as regras impostas pelo sistema"""
  # await asyncio.sleep(3)
  response =  await agent_policy.ainvoke({"messages": [HumanMessage(content=f"Me diga se a pergunta desse usuário viola alguma das regras do sistema. {query}")]})
  return response["messages"][-1].content


@tool
async def call_retriver_subagent(query: str) -> str:
  """Chama o subagente 2 que busca os documentos mais relevantes de acordo com a query"""
  # await asyncio.sleep(3)
  response = await agent_retriver.ainvoke({"messages": [HumanMessage(content=f"Me retorne os documentos mais relevantes de acordo com essa query {query}")]})
  return response["messages"][-1].content



@tool
async def call_qa_subagent(query: str, docs: str) -> str:
  """Chama o subagente 3 que estrutura uma resposta de acordo com a pergunta e os documentos passados"""
  # await asyncio.sleep(3)
  response = await agent_qa.ainvoke({"messages": [HumanMessage(content=f"Responda a pergunta com base nos documentos fornecidos. Pergunta: {query}. Documentos recuperados: {docs}")]})
  return response["messages"][-1].content

@tool
async def call_selfcheck_subagent(query: str, docs: str, response: str) -> str:
  """Chama o subagente 4 que valida a resposta do usuário com base em regras pré definidas e decide se é válida, necessita de re-busca ou retornar uma mensagem de pergunta inválida"""
  # await asyncio.sleep(3)
  response_agent = await agent_check.ainvoke({"messages": [HumanMessage(content=f"Valide a resposta do qa com base nas regras pré-definidas. Pergunta: {query}. Resposta: {response}. Documentos recuperados: {docs}")]})
  return response_agent["messages"][-1].content

@tool
async def call_logging_automation(pergunta: str, resposta: str, veredito: str) -> str:
    """Registra a interação aluno-IA usando logs locais."""
    
    # O SEGREDO: Use apenas o nome do arquivo. 
    # O MCP já sabe que deve salvar dentro de /home/.../logs_professor/
    filename = "log_interacao.txt" 
    
    log_content = f"Pergunta: {pergunta}\nResposta: {resposta}\nVeredito: {veredito}"
    
    # Instrução ultra-específica para o Llama 3 não inventar caminhos
    instrucao = f"Escreva EXATAMENTE no arquivo '{filename}' o seguinte conteúdo: {log_content}"
    
    await agent_automation.ainvoke({"messages": [HumanMessage(content=instrucao)]})
    return "Log registrado com sucesso."


In [19]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

In [ ]:
agent_supervisor = create_agent(model=model, tools=[call_retriver_subagent, call_policy_subagent, call_qa_subagent, call_selfcheck_subagent], system_prompt=supervisor_prompt[-1].content, 
                                checkpointer=InMemorySaver(),
                                middleware=[SummarizationMiddleware(
                                    model=model,
                                    trigger=('tokens',100),
                                    keep=("messages",1)
                                )])


In [21]:
from pprint import pprint

In [47]:
results = []
for tool in [call_policy_subagent, call_retriver_subagent, call_qa_subagent,call_logging_automation]:
    results.append(tool.name)
    results.append(tool.args_schema.schema())


In [44]:
from langchain_core.messages import HumanMessage  

config = {"configurable": {"thread_id": "teste_novo_caminho"}}

response = await agent_supervisor.ainvoke(
    {"messages": [HumanMessage(content="What is concurrency in the context of Programming?")]},
    config=config
)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khwjg6grety8p5zjz7gytzd5` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99987, Requested 492. Please try again in 6m53.856s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [34]:
response

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nExplain the concept of concurrency in programming.\n\n## SUMMARY\nThe user is asking for a definition and explanation of concurrency in programming.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nProvide a clear and concise explanation of concurrency in programming.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='cf315bd9-d7e5-4a52-af81-c6cd7da37eff'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'call_policy_subagent', 'arguments': '{"query": "What is concurrency in the context of Programming?"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf98e-b17d-76a1-91fc-b050cb7302d0-0', tool_calls=[{'name': 'call_policy_subagent', 'args': {'query': 'What is concurrency in the context of Programming?'}, 'id': 'c7c420df-4d71-436e-9abb-

In [ ]:
# Verifique se está assim:
LOG_DIR = os.path.abspath("./logs_professor")
# ...
print(LOG_DIR)

/home/jardel/Documentos/UFCG/llm/projeto_final/Projeto-LLM/logs_professor
